# nodes.fetch_adzuna

Download raw Adzuna job ad JSONL files from S3, insert into DuckDB,
and deduplicate by ID (keeping the row with the earliest `date_created`).
Hive-partitioned: `s3://{prefix}/year={Y}/month={M}/day={D}/*.jsonl`

In [ ]:
#|default_exp fetch_adzuna
#|export_as_func true

In [ ]:
#|set_func_signature
def main(ctx, print):
    """Download raw Adzuna job ads from S3, insert into DuckDB, and deduplicate."""
    ...


Retrieve input arguments

In [ ]:
from dev_utils import *
run_name = 'test_local'
set_node_func_args('fetch_adzuna', run_name=run_name)
show_node_vars('fetch_adzuna', run_name=run_name)

# Function body

In [ ]:
#|export
# Skip fetch if data is already ingested (avoids DuckDB contention in concurrent runs)
skip_fetch = ctx.vars["skip_fetch"]
if skip_fetch:
    print("fetch_adzuna: skip_fetch=true, assuming data already ingested")
    None #|func_return_line

import json
import os
import shutil
import tempfile

import boto3
import pyarrow.json as paj
import pyarrow.parquet as pq

from ai_index.utils import get_adzuna_conn, ensure_ads_table, build_insert_from_parquet

s3_prefix = ctx.vars["adzuna_s3_prefix"]
years_filter = ctx.vars["fetch_years"]
duckdb_memory_limit = ctx.vars["duckdb_memory_limit"]

# Parse bucket and key prefix from s3_prefix (format: "bucket/key/prefix")
bucket_name, _, key_prefix = s3_prefix.partition("/")
s3 = boto3.client("s3")

# Discover all year partitions from S3
import re
paginator = s3.get_paginator("list_objects_v2")
years = set()
for page in paginator.paginate(Bucket=bucket_name, Prefix=f"{key_prefix}/year=", Delimiter="/"):
    for prefix_obj in page.get("CommonPrefixes", []):
        m = re.search(r"year=(\d+)", prefix_obj["Prefix"])
        if m:
            years.add(m.group(1))
years = sorted(years)
print(f"fetch_adzuna: discovered {len(years)} year(s) in S3: {years}")

# Filter to requested years
if years_filter and years_filter != "all":
    requested = {y.strip() for y in years_filter.split(",")}
    years = [y for y in years if y in requested]
    print(f"fetch_adzuna: filtered to {len(years)} year(s): {years}")

conn = get_adzuna_conn(memory_limit=duckdb_memory_limit)
ensure_ads_table(conn)

adzuna_meta = {"years": {}}
any_new_data = False

for year in years:
    year_int = int(year)
    year_info = {"months": [], "row_counts": {}, "new_months": []}

    for month in range(1, 13):
        # Idempotent: skip if this year/month already has rows in DuckDB
        existing = conn.execute(
            "SELECT COUNT(*) FROM ads WHERE year = ? AND month = ?",
            [year_int, month],
        ).fetchone()[0]
        if existing > 0:
            year_info["months"].append(month)
            year_info["row_counts"][month] = existing
            print(f"fetch_adzuna: {year}/month_{month:02d} already in DB ({existing} rows), skipping")
            continue

        # List all JSON/JSONL objects under this month's Hive partition
        month_key = f"{key_prefix}/year={year}/month={month}/"
        paginator = s3.get_paginator("list_objects_v2")
        data_keys = []
        for page in paginator.paginate(Bucket=bucket_name, Prefix=month_key):
            for obj in page.get("Contents", []):
                if obj["Key"].endswith(".json") or obj["Key"].endswith(".jsonl"):
                    data_keys.append(obj["Key"])

        if not data_keys:
            print(f"fetch_adzuna: no data files for {year}/month={month}, skipping")
            continue

        # Download each S3 file to temp, convert to parquet, then INSERT into DuckDB.
        tmp_dir = tempfile.mkdtemp()
        total_rows = 0
        n_files = len(data_keys)

        conn.execute("BEGIN TRANSACTION")
        try:
            for i, key in enumerate(sorted(data_keys)):
                print(f"fetch_adzuna: [{i+1}/{n_files}] downloading s3://{bucket_name}/{key}")
                tmp_json = f"{tmp_dir}/day_{i}.json"
                s3.download_file(bucket_name, key, tmp_json)
                # Use pyarrow's native JSON reader (columnar, parallelized C++)
                read_opts = paj.ReadOptions(block_size=1 << 26)  # 64 MB blocks
                table = paj.read_json(tmp_json, read_options=read_opts)
                tmp_pq = f"{tmp_dir}/day_{i}.parquet"
                pq.write_table(table, tmp_pq, compression="snappy")
                n_rows = table.num_rows
                total_rows += n_rows
                print(f"  -> {n_rows} rows")
                del table
                os.remove(tmp_json)

                # INSERT into DuckDB from temp parquet
                insert_sql = build_insert_from_parquet(tmp_pq, year_int, month)
                conn.execute(insert_sql)
                os.remove(tmp_pq)

            conn.execute("COMMIT")
        except Exception:
            conn.execute("ROLLBACK")
            raise
        finally:
            shutil.rmtree(tmp_dir, ignore_errors=True)

        if total_rows > 0:
            year_info["months"].append(month)
            year_info["new_months"].append(month)
            year_info["row_counts"][month] = total_rows
            print(f"fetch_adzuna: inserted {year}/month_{month:02d} ({total_rows} rows)")
        else:
            print(f"fetch_adzuna: no data found for {year}/month={month}")

    # --- Dedup: keep earliest date_created per id within year ---
    months_fingerprint = ",".join(str(m) for m in sorted(year_info["months"]))
    meta_key = f"dedup_{year}"

    needs_dedup = bool(year_info["new_months"])
    if not needs_dedup:
        # No new months. Check if dedup already ran for this exact set
        existing_marker = conn.execute(
            "SELECT value FROM _meta WHERE key = ?", [meta_key]
        ).fetchone()
        if existing_marker:
            marker = json.loads(existing_marker[0])
            if marker["months_fingerprint"] == months_fingerprint:
                print(f"fetch_adzuna: {year} already deduplicated (months match), skipping dedup")
                year_info["duplicates_removed"] = marker["duplicates_removed"]
            else:
                needs_dedup = True
        # If no marker and no new months, nothing to dedup

    if needs_dedup and year_info["months"]:
        before_count = conn.execute(
            "SELECT COUNT(*) FROM ads WHERE year = ?", [year_int]
        ).fetchone()[0]

        # Keep the row with the earliest date_created per id (rowid as tiebreaker)
        conn.execute("""
            DELETE FROM ads
            WHERE year = ?
              AND rowid NOT IN (
                  SELECT arg_min(rowid, date_created)
                  FROM ads
                  WHERE year = ?
                  GROUP BY id
              )
        """, [year_int, year_int])

        after_count = conn.execute(
            "SELECT COUNT(*) FROM ads WHERE year = ?", [year_int]
        ).fetchone()[0]
        dups_removed = before_count - after_count
        year_info["duplicates_removed"] = dups_removed

        # Update per-month row counts after dedup
        for month in year_info["months"]:
            year_info["row_counts"][month] = conn.execute(
                "SELECT COUNT(*) FROM ads WHERE year = ? AND month = ?",
                [year_int, month],
            ).fetchone()[0]

        # Write dedup marker
        marker_data = {
            "months_fingerprint": months_fingerprint,
            "duplicates_removed": dups_removed,
        }
        conn.execute(
            "INSERT OR REPLACE INTO _meta (key, value) VALUES (?, ?)",
            [meta_key, json.dumps(marker_data)],
        )

        conn.execute("CHECKPOINT")
        print(f"fetch_adzuna: {year} dedup, {dups_removed} duplicates removed ({before_count} -> {after_count} rows)")

    if year_info["new_months"]:
        any_new_data = True
    del year_info["new_months"]
    adzuna_meta["years"][year] = year_info
    total_rows = sum(year_info["row_counts"].values())
    print(f"fetch_adzuna: year {year}, {len(year_info['months'])} months, {total_rows} total rows")

# --- Global dedup: remove cross-year duplicates (only if new data was ingested) ---
if any_new_data:
    before_total = conn.execute("SELECT COUNT(*) FROM ads").fetchone()[0]
    conn.execute("""
        DELETE FROM ads
        WHERE rowid NOT IN (
            SELECT arg_min(rowid, date_created)
            FROM ads
            GROUP BY id
        )
    """)
    after_total = conn.execute("SELECT COUNT(*) FROM ads").fetchone()[0]
    global_dups = before_total - after_total
    if global_dups > 0:
        conn.execute("CHECKPOINT")
        print(f"fetch_adzuna: global dedup, {global_dups} cross-year duplicates removed ({before_total} -> {after_total} rows)")
    else:
        print(f"fetch_adzuna: global dedup, no cross-year duplicates")
else:
    print(f"fetch_adzuna: no new data ingested, skipping global dedup")

conn.close()
print(f"fetch_adzuna: done, {len(adzuna_meta['years'])} year(s)")